Task1

In [30]:
import os
import random
import urllib.request
import torch
import torch.nn.functional as F

In [31]:
words = open("names.txt", "r").read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s: i + 1 for i, s in enumerate(chars)}
stoi["."] = 0
itos = {i: s for s, i in stoi.items()}
vocab_size = len(itos)  # 27


In [32]:
block_size = 3  # kac karakterlik baglama bakiyoruz


def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + ".":
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    return torch.tensor(X), torch.tensor(Y)
 #... ---> e
 #..e ---> m

random.seed(42)
random.shuffle(words)
n1 = int(0.8 * len(words))
n2 = int(0.9 * len(words))
Xtr, Ytr = build_dataset(words[:n1])
Xdev, Ydev = build_dataset(words[n1:n2])
Xte, Yte = build_dataset(words[n2:])
print(f"kelime sayisi: {len(words)} | vocab: {vocab_size} | Xtr: {tuple(Xtr.shape)}")

kelime sayisi: 32033 | vocab: 27 | Xtr: (182625, 3)


In [33]:
results = []


def cmp(s, dt, t):
    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    results.append((s, ex, app))
    print(f"{s:15s} | exact: {str(ex):5s} | approximate: {str(app):5s} | maxdiff: {maxdiff}")

    # dt elle hesapladığımız gradyanlar
    # t ise backward ün hesapladığı gradyan
    #sonuçları kıyaslıyoruz cmp sayesinde

In [34]:
n_embd = 10    # karakter embedding boyutu
n_hidden = 64  # gizli katmandaki noron sayisi

g = torch.Generator().manual_seed(2147483647)
C = torch.randn((vocab_size, n_embd), generator=g)
# Katman 1
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5 / 3) / ((n_embd * block_size) ** 0.5)
b1 = torch.randn(n_hidden, generator=g) * 0.1  # BN yuzunden ise yaramaz; gradyan kontrolu icin tutuluyor
# Katman 2
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.1
b2 = torch.randn(vocab_size, generator=g) * 0.1
# BatchNorm parametreleri
bngain = torch.randn((1, n_hidden)) * 0.1 + 1.0
bnbias = torch.randn((1, n_hidden)) * 0.1


parameters = [C, W1, b1, W2, b2, bngain, bnbias]
print("toplam parametre:", sum(p.nelement() for p in parameters))
for p in parameters:
    p.requires_grad = True

batch_size = 32
n = batch_size
ix = torch.randint(0, Xtr.shape[0], (batch_size,), generator=g)
Xb, Yb = Xtr[ix], Ytr[ix]

toplam parametre: 4137


In [35]:
 #TASK 1: forward pass'i atomik adimlara bol + loss.backward()

emb = C[Xb]                                  # (32, 3, 10)  embedding lookup
embcat = emb.view(emb.shape[0], -1)          # (32, 30)     vektorleri yan yana koy
# Lineer katman 1
hprebn = embcat @ W1 + b1                    # (32, 64)
# BatchNorm (tek tek parcalara bolunmus hali)
bnmeani = 1 / n * hprebn.sum(0, keepdim=True)            # (1, 64)  ortalama
bndiff = hprebn - bnmeani                                # (32, 64)
bndiff2 = bndiff ** 2                                    # (32, 64)
bnvar = 1 / (n - 1) * (bndiff2).sum(0, keepdim=True)     # (1, 64)  Bessel duzeltmesi: n-1
bnvar_inv = (bnvar + 1e-5) ** -0.5                       # (1, 64)
bnraw = bndiff * bnvar_inv                               # (32, 64)
hpreact = bngain * bnraw + bnbias                        # (32, 64)
# Non-lineerlik
h = torch.tanh(hpreact)                      # (32, 64)
# Lineer katman 2
logits = h @ W2 + b2                         # (32, 27)
# Cross entropy (F.cross_entropy(logits, Yb) ile ayni sey, ama adim adim yapacağız)
logit_maxes = logits.max(1, keepdim=True).values         # (32, 1)
norm_logits = logits - logit_maxes                       # (32, 27) max değeri diğer değerlerden çıkartıp normalize ediyoruz
counts = norm_logits.exp()                               # (32, 27)
counts_sum = counts.sum(1, keepdim=True)                 # (32, 1)
counts_sum_inv = counts_sum ** -1                        # (32, 1)
probs = counts * counts_sum_inv                          # (32, 27)
logprobs = probs.log()                                   # (32, 27)
loss = -logprobs[range(n), Yb].mean()                    # skaler

# PyTorch backward: ara degiskenlerin gradyani normalde saklanmaz -> retain_grad()
intermediates = {
    "logprobs": logprobs, "probs": probs, "counts": counts, "counts_sum": counts_sum,
    "counts_sum_inv": counts_sum_inv, "norm_logits": norm_logits, "logit_maxes": logit_maxes,
    "logits": logits, "h": h, "hpreact": hpreact, "bnraw": bnraw, "bnvar_inv": bnvar_inv,
    "bnvar": bnvar, "bndiff2": bndiff2, "bndiff": bndiff, "hprebn": hprebn,
    "bnmeani": bnmeani, "embcat": embcat, "emb": emb,
}
for p in parameters:
    p.grad = None
for t in intermediates.values():
    t.retain_grad()
loss.backward()

print("\n" + "=" * 78)
print("TASK 1 - loss.backward() ile elde edilen gradyanlar")
print("=" * 78)
print(f"loss = {loss.item():.4f}   (F.cross_entropy ile fark: "
      f"{(F.cross_entropy(logits, Yb) - loss).abs().item():.2e})\n")
print(f"{'degisken':15s} | {'shape':12s} | {'grad shape':12s} | {'|grad| ort.':>12s} | {'|grad| max':>12s}")
print("-" * 78)
param_names = {"C": C, "W1": W1, "b1": b1, "W2": W2, "b2": b2, "bngain": bngain, "bnbias": bnbias}
for name, t in list(intermediates.items()) + list(param_names.items()):
    gr = t.grad
    print(f"{name:15s} | {str(tuple(t.shape)):12s} | {str(tuple(gr.shape)):12s} | "
          f"{gr.abs().mean().item():12.3e} | {gr.abs().max().item():12.3e}")



TASK 1 - loss.backward() ile elde edilen gradyanlar
loss = 3.3206   (F.cross_entropy ile fark: 4.77e-07)

degisken        | shape        | grad shape   |  |grad| ort. |   |grad| max
------------------------------------------------------------------------------
logprobs        | (32, 27)     | (32, 27)     |    1.157e-03 |    3.125e-02
probs           | (32, 27)     | (32, 27)     |    3.716e-02 |    2.451e+00
counts          | (32, 27)     | (32, 27)     |    6.446e-03 |    2.733e-01
counts_sum      | (32, 1)      | (32, 1)      |    3.039e-03 |    5.718e-03
counts_sum_inv  | (32, 1)      | (32, 1)      |    3.419e-01 |    5.421e-01
norm_logits     | (32, 27)     | (32, 27)     |    2.216e-03 |    3.085e-02
logit_maxes     | (32, 1)      | (32, 1)      |    2.321e-09 |    7.451e-09
logits          | (32, 27)     | (32, 27)     |    2.216e-03 |    3.085e-02
h               | (32, 64)     | (32, 64)     |    2.592e-03 |    1.107e-02
hpreact         | (32, 64)     | (32, 64)     |    1.5

**Task2**

In [36]:
#TASK 2: ayni gradyanlari elle yaz (loss'tan geriye dogru)
# ============================================================================
# Surekli kullanilan uc kural:
#   (a) dX'in shape'i her zaman X'in shape'i ile aynidir.
#   (b) forward'da broadcast (kopyalama) olduysa backward'da o eksende SUM alinir;
#       forward'da sum varsa backward'da gradyan o eksene dagitilir (ones_like * ...).
#   (c) bir degisken grafikte iki yere gidiyorsa gradyanlari toplanir (+=).

# --- cross entropy kismi ----------------------------------------------------
# loss = -(1/n) * sum_i logprobs[i, Yb[i]]  -> sadece dogru sinifin hucreleri -1/n alir
dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(n), Yb] = -1.0 / n

# logprobs = log(probs)  ->  d/dprobs = 1/probs
dprobs = (1.0 / probs) * dlogprobs

# probs = counts * counts_sum_inv ; counts_sum_inv (32,1) -> (32,27)'ye broadcast oluyor
# broadcast edilen tarafin gradyani satir boyunca toplanir (kural b)
dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
dcounts = counts_sum_inv * dprobs                        # counts'un 1. dali

# counts_sum_inv = counts_sum**-1  ->  turev: -counts_sum**-2
dcounts_sum = (-counts_sum ** -2) * dcounts_sum_inv

# counts_sum = counts.sum(1)  ->  toplamanin turevi 1; gradyan satirdaki her elemana aynen dagilir
dcounts += torch.ones_like(counts) * dcounts_sum         # counts'un 2. dali (kural c)

# counts = exp(norm_logits)  ->  exp'in turevi kendisi (= counts)
dnorm_logits = counts * dcounts

# norm_logits = logits - logit_maxes ; logit_maxes (32,1) broadcast -> satir boyunca sum, isaret eksi
dlogits = dnorm_logits.clone()                           # logits'in 1. dali
dlogit_maxes = (-dnorm_logits).sum(1, keepdim=True)      # ~0 cikar: max cikarmak probs'u degistirmez

# logit_maxes = logits.max(1)  ->  gradyan sadece max'in oldugu pozisyona akar (one-hot maske)
dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes  # 2. dal

# --- lineer katman 2:  logits = h @ W2 + b2 ----------------------------------
# shape'leri tutturmanin tek yolu:
dh = dlogits @ W2.T              # (32,27) @ (27,64) -> (32,64)
dW2 = h.T @ dlogits              # (64,32) @ (32,27) -> (64,27)
db2 = dlogits.sum(0)             # b2 (27,) batch boyunca broadcast oldu -> eksen 0'da sum

# --- tanh:  h = tanh(hpreact)  ->  turev: 1 - tanh^2 = 1 - h^2 ---------------
dhpreact = (1.0 - h ** 2) * dh

# --- batchnorm: hpreact = bngain * bnraw + bnbias ----------------------------
dbngain = (bnraw * dhpreact).sum(0, keepdim=True)        # (1,64) broadcast oldu -> eksen 0'da sum
dbnraw = bngain * dhpreact
dbnbias = dhpreact.sum(0, keepdim=True)

# bnraw = bndiff * bnvar_inv ; bnvar_inv (1,64) broadcast
dbndiff = bnvar_inv * dbnraw                             # bndiff'in 1. dali
dbnvar_inv = (bndiff * dbnraw).sum(0, keepdim=True)

# bnvar_inv = (bnvar + eps)**-0.5  ->  turev: -0.5 * (bnvar + eps)**-1.5
dbnvar = (-0.5 * (bnvar + 1e-5) ** -1.5) * dbnvar_inv

# bnvar = 1/(n-1) * bndiff2.sum(0)  ->  her elemana 1/(n-1) katsayisiyla dagilir
dbndiff2 = (1.0 / (n - 1)) * torch.ones_like(bndiff2) * dbnvar

# bndiff2 = bndiff**2  ->  turev: 2*bndiff
dbndiff += (2 * bndiff) * dbndiff2                       # bndiff'in 2. dali

# bndiff = hprebn - bnmeani ; bnmeani (1,64) broadcast
dhprebn = dbndiff.clone()                                # hprebn'in 1. dali
dbnmeani = (-dbndiff).sum(0, keepdim=True)

# bnmeani = 1/n * hprebn.sum(0)  ->  her satira 1/n katsayisiyla dagilir
dhprebn += 1.0 / n * (torch.ones_like(hprebn) * dbnmeani)  # hprebn'in 2. dali

In [37]:
# --- lineer katman 1:  hprebn = embcat @ W1 + b1 -----------------------------
dembcat = dhprebn @ W1.T         # (32,64) @ (64,30) -> (32,30)
dW1 = embcat.T @ dhprebn         # (30,32) @ (32,64) -> (30,64)
db1 = dhprebn.sum(0)

# --- embcat = emb.view(32, -1)  ->  sadece yeniden sekillendirme, geri cevir --
demb = dembcat.view(emb.shape)

# --- emb = C[Xb]  ->  her (k,j) pozisyonunun gradyani, kullanilan C satirina eklenir
# ayni karakter batch'te birden cok kez gecebildigi icin += ile birikir
dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix_ = Xb[k, j]
        dC[ix_] += demb[k, j]

print("\n" + "=" * 78)
print("TASK 2 - elle yazilan gradyanlar vs PyTorch (cmp)")
print("=" * 78)
cmp("logprobs", dlogprobs, logprobs)
cmp("probs", dprobs, probs)
cmp("counts_sum_inv", dcounts_sum_inv, counts_sum_inv)
cmp("counts_sum", dcounts_sum, counts_sum)
cmp("counts", dcounts, counts)
cmp("norm_logits", dnorm_logits, norm_logits)
cmp("logit_maxes", dlogit_maxes, logit_maxes)
cmp("logits", dlogits, logits)
cmp("h", dh, h)
cmp("W2", dW2, W2)
cmp("b2", db2, b2)
cmp("hpreact", dhpreact, hpreact)
cmp("bngain", dbngain, bngain)
cmp("bnbias", dbnbias, bnbias)
cmp("bnraw", dbnraw, bnraw)
cmp("bnvar_inv", dbnvar_inv, bnvar_inv)
cmp("bnvar", dbnvar, bnvar)
cmp("bndiff2", dbndiff2, bndiff2)
cmp("bndiff", dbndiff, bndiff)
cmp("bnmeani", dbnmeani, bnmeani)
cmp("hprebn", dhprebn, hprebn)
cmp("embcat", dembcat, embcat)
cmp("W1", dW1, W1)
cmp("b1", db1, b1)
cmp("emb", demb, emb)
cmp("C", dC, C)

n_exact = sum(1 for _, ex, _ in results if ex)
n_approx = sum(1 for _, ex, app in results if (not ex) and app)
n_fail = sum(1 for _, ex, app in results if (not ex) and (not app))
print("-" * 78)
print(f"toplam {len(results)} gradyan | exact: {n_exact} | sadece approximate: {n_approx} | hatali: {n_fail}")
print(f"torch surumu: {torch.__version__}")


TASK 2 - elle yazilan gradyanlar vs PyTorch (cmp)
logprobs        | exact: True  | approximate: True  | maxdiff: 0.0
probs           | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum_inv  | exact: True  | approximate: True  | maxdiff: 0.0
counts_sum      | exact: True  | approximate: True  | maxdiff: 0.0
counts          | exact: True  | approximate: True  | maxdiff: 0.0
norm_logits     | exact: True  | approximate: True  | maxdiff: 0.0
logit_maxes     | exact: True  | approximate: True  | maxdiff: 0.0
logits          | exact: True  | approximate: True  | maxdiff: 0.0
h               | exact: True  | approximate: True  | maxdiff: 0.0
W2              | exact: True  | approximate: True  | maxdiff: 0.0
b2              | exact: True  | approximate: True  | maxdiff: 0.0
hpreact         | exact: False | approximate: True  | maxdiff: 4.656612873077393e-10
bngain          | exact: False | approximate: True  | maxdiff: 1.862645149230957e-09
bnbias          | exact: False | approxima